<a href="https://colab.research.google.com/github/Christi049/Smart-Pantry/blob/main/Grocery_saver_agent_finetunedModel_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

file = json.load(open("Dataset.json", "r"))
print(file[1])

{'instruction': 'Suggest a weekly meal plan for 4 people, prioritizing ingredients that will expire soon. The meal plan must strictly adhere to the specified dietary restrictions and be free of all listed allergens. The plan should include three meals per day, with varied dishes, and minimal extra shopping. Leftovers should be used to reduce cooking effort and food waste.', 'input': {'serving_size': 4, 'dietary_restrictions': ['Vegetarian'], 'allergens': ['None'], 'ingredients': [{'name': 'Paneer', 'quantity': '500g', 'expiry': '2025-09-04'}, {'name': 'Bell peppers', 'quantity': '4 large', 'expiry': '2025-09-05'}, {'name': 'Onions', 'quantity': '6 medium', 'expiry': '2025-09-10'}, {'name': 'Tomatoes', 'quantity': '6 large', 'expiry': '2025-09-06'}, {'name': 'Yogurt', 'quantity': '300g', 'expiry': '2025-09-03'}, {'name': 'Milk', 'quantity': '1000ml', 'expiry': '2025-09-03'}]}, 'output': {'meals': {'Monday': {'breakfast': {'description_for_model': 'Lassi (Yogurt drink) using the full 300

In [ ]:
!pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
!pip install -q huggingface_hub transformers gradio accelerate

In [ ]:
# Only if you want to include your existing examples
with open("Dataset.json", "r") as f:
    current_examples = json.load(f)

In [ ]:
import random
import json
import itertools

def generate_accurate_examples(num_examples):
    # A more extensive list of ingredients and dietary information
    all_ingredients = {
        "eggs": {"vegan": False, "dairy": False, "gluten": True},
        "cheese": {"vegan": False, "dairy": True, "gluten": True},
        "spinach": {"vegan": True, "dairy": False, "gluten": True},
        "tofu": {"vegan": True, "dairy": False, "gluten": True, "soy": True},
        "bread": {"vegan": True, "dairy": False, "gluten": False},
        "chicken": {"vegan": False, "dairy": False, "gluten": True},
        "lentils": {"vegan": True, "dairy": False, "gluten": False},
        "coconut milk": {"vegan": True, "dairy": False, "gluten": True},
        "rice": {"vegan": True, "dairy": False, "gluten": False},
        "potatoes": {"vegan": True, "dairy": False, "gluten": True},
        "onions": {"vegan": True, "dairy": False, "gluten": True},
        "bell peppers": {"vegan": True, "dairy": False, "gluten": True},
        "nuts": {"vegan": True, "dairy": False, "gluten": True, "nuts": True},
        "avocado": {"vegan": True, "dairy": False, "gluten": True}  # ADDED TO FIX KEYERROR
    }

    # A simple, rule-based recipe database for a single meal
    recipes = {
        "Breakfast": {
            "Omelette": ["eggs", "cheese", "spinach"],
            "Scrambled Tofu": ["tofu", "spinach", "onions"],
            "Toast with Avocado": ["bread", "avocado"]
        },
        "Lunch": {
            "Chicken and Veggies": ["chicken", "rice", "bell peppers"],
            "Lentil Soup": ["lentils", "onions", "carrots"],
            "Veggie Burger": ["lentils", "bread", "spinach"]
        },
        "Dinner": {
            "Chicken Curry": ["chicken", "coconut milk", "rice"],
            "Spinach and Potatoes": ["potatoes", "spinach", "onions"],
            "Vegan Stir-fry": ["tofu", "rice", "bell peppers"]
        }
    }

    dietary_options = ["None", "Vegetarian", "Vegan", "Gluten-Free"]
    allergen_options = ["None", "Nuts", "Dairy", "Soy", "Gluten"]
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    new_examples = []

    for _ in range(num_examples):
        # 1. Randomly generate input parameters for the example
        serving_size = random.randint(1, 4)
        dietary_restriction = random.choice(dietary_options)
        allergens = random.sample(allergen_options, k=random.randint(0, 2))
        available_ingredients_list = random.sample(list(all_ingredients.keys()), k=random.randint(3, 6))

        # 2. Dynamically generate the output (meal plan and shopping list)
        meal_plan = {}
        shopping_list = set()

        for day in days:
            meals = {}
            for meal_type, meal_recipes in recipes.items():
                valid_recipes = []
                for recipe_name, required_ingredients in meal_recipes.items():
                    is_compatible = True
                    for ingredient in required_ingredients:
                        # Check if ingredient is in our master list before trying to access it.
                        if ingredient not in all_ingredients:
                            is_compatible = False
                            break

                        # Check for general dietary restrictions
                        if dietary_restriction == "Vegetarian" and all_ingredients[ingredient]["vegan"] == False:
                            is_compatible = False
                            break
                        if dietary_restriction == "Vegan" and all_ingredients[ingredient]["vegan"] == False:
                            is_compatible = False
                            break

                        # Check for allergens by using .get() or 'in' keyword to avoid KeyError
                        for allergen in allergens:
                            if all_ingredients[ingredient].get(allergen, False) == True:
                                is_compatible = False
                                break

                        if not is_compatible:
                            break

                    if is_compatible:
                        valid_recipes.append((recipe_name, required_ingredients))

                if not valid_recipes:
                    meals[meal_type] = {"description_for_model": "Simple meal", "display_text": "Simple meal"}
                    continue

                chosen_recipe_name, chosen_ingredients = random.choice(valid_recipes)

                for ingredient in chosen_ingredients:
                    if ingredient not in available_ingredients_list:
                        shopping_list.add(ingredient)

                ingredient_str = ", ".join(chosen_ingredients)
                description = f"Prepare {chosen_recipe_name} with {ingredient_str}. This is a {dietary_restriction} and {', '.join(allergens)} free meal."
                meals[meal_type] = {
                    "description_for_model": description,
                    "display_text": chosen_recipe_name
                }
            meal_plan[day] = meals

        # 3. Assemble the final example
        instruction = (
            f"Suggest a weekly meal plan for {serving_size} person(s), prioritizing "
            "already available ingredients. The plan must adhere to the specified dietary "
            "restrictions and be free of all listed allergens. Also provide a shopping list for any items not present."
        )
        input_data = {
            "serving_size": serving_size,
            "dietary_restrictions": [dietary_restriction],
            "allergens": allergens,
            "ingredients": [{"name": i, "quantity": f"{random.randint(100, 500)}g", "expiry": "2025-09-03"} for i in available_ingredients_list]
        }
        output_data = {
            "meals": meal_plan,
            "shopping_list": list(shopping_list)
        }

        new_examples.append({
            "instruction": instruction,
            "input": input_data,
            "output": output_data
        })

    return new_examples
new_examples = generate_accurate_examples(320)

In [ ]:
# Only if you loaded current_examples
combined_dataset = current_examples + new_examples

In [ ]:
# Save new dataset only
with open("dataset.json", "w") as f:
    json.dump(new_examples, f, indent=2)

# Save combined dataset (if combined)
with open("combined_dataset.json", "w") as f:
    json.dump(combined_dataset, f, indent=2)


In [ ]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

max_seq_length = 2048  # Choose sequence length
dtype = None  # Auto detection

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
import json
from datasets import Dataset

def format_prompt(example):
    return (
        f"### Instruction: {example['instruction']}\n"
        f"### Input: {json.dumps(example['input'], ensure_ascii=False)}\n"
        f"### Output: {json.dumps(example['output'], ensure_ascii=False)}<|endoftext|>"
    )

# Use your combined_dataset instead of 'file'
formatted_data = [format_prompt(item) for item in combined_dataset]

# Create a Hugging Face Dataset
dataset = Dataset.from_dict({"text": formatted_data})

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,  # LoRA rank - higher = more capacity, more memory
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,  # LoRA scaling factor (usually 2x rank)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,          # replaces tokenizer=
    train_dataset=dataset,

    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        logging_steps=25,

        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",

        seed=3407,
        output_dir="outputs",

        save_strategy="epoch",
        save_total_limit=2,

        dataloader_pin_memory=False,
        report_to="none",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/428 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 428 | Num Epochs = 3 | Total steps = 162
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
25,0.280820
50,0.122977
75,0.104569
100,0.096335
125,0.078585
150,0.070526


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-54/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-108/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-162/tokenizer_config.json.


In [ ]:
# ============================================================
# SMARTPANTRY - MODEL EVALUATION
# ============================================================

import json
import re
import random
import pandas as pd
import torch
from datetime import datetime

# ------------------------------------------------------------
# 1. Evaluation test cases
# ------------------------------------------------------------

test_cases = [
    {
        "serving_size": 2,
        "dietary_restrictions": ["Vegan"],
        "allergens": ["Nuts"],
        "ingredients": [
            {"name": "Lentils", "quantity": "250g", "expiry": "2026-09-18"},
            {"name": "Tofu", "quantity": "200g", "expiry": "2026-09-19"},
            {"name": "Spinach", "quantity": "1 bunch", "expiry": "2026-09-20"},
            {"name": "Cabbage", "quantity": "1 medium", "expiry": "2026-09-24"},
            {"name": "Potatoes", "quantity": "4 medium", "expiry": "2026-09-27"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-10-01"}
        ]
    },

    {
        "serving_size": 3,
        "dietary_restrictions": ["Vegetarian"],
        "allergens": ["Nuts"],
        "ingredients": [
            {"name": "Paneer", "quantity": "400g", "expiry": "2026-09-18"},
            {"name": "Yogurt", "quantity": "300g", "expiry": "2026-09-19"},
            {"name": "Tomatoes", "quantity": "6", "expiry": "2026-09-21"},
            {"name": "Bell peppers", "quantity": "4", "expiry": "2026-09-23"},
            {"name": "Onions", "quantity": "5", "expiry": "2026-09-25"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-10-01"}
        ]
    },

    {
        "serving_size": 2,
        "dietary_restrictions": ["Vegan"],
        "allergens": ["Soy"],
        "ingredients": [
            {"name": "Chickpeas", "quantity": "300g", "expiry": "2026-09-18"},
            {"name": "Spinach", "quantity": "1 bunch", "expiry": "2026-09-19"},
            {"name": "Potatoes", "quantity": "4", "expiry": "2026-09-21"},
            {"name": "Tomatoes", "quantity": "5", "expiry": "2026-09-23"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-09-30"}
        ]
    },

    {
        "serving_size": 4,
        "dietary_restrictions": ["Vegetarian"],
        "allergens": ["Dairy"],
        "ingredients": [
            {"name": "Lentils", "quantity": "500g", "expiry": "2026-09-18"},
            {"name": "Chickpeas", "quantity": "400g", "expiry": "2026-09-20"},
            {"name": "Potatoes", "quantity": "5", "expiry": "2026-09-22"},
            {"name": "Spinach", "quantity": "2 bunches", "expiry": "2026-09-24"},
            {"name": "Rice", "quantity": "1kg", "expiry": "2026-10-01"}
        ]
    },

    {
        "serving_size": 2,
        "dietary_restrictions": ["Non-Vegetarian"],
        "allergens": ["Nuts"],
        "ingredients": [
            {"name": "Chicken", "quantity": "500g", "expiry": "2026-09-18"},
            {"name": "Potatoes", "quantity": "4", "expiry": "2026-09-20"},
            {"name": "Tomatoes", "quantity": "5", "expiry": "2026-09-22"},
            {"name": "Onions", "quantity": "5", "expiry": "2026-09-24"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-09-30"}
        ]
    },

    {
        "serving_size": 2,
        "dietary_restrictions": ["Vegan"],
        "allergens": ["Gluten"],
        "ingredients": [
            {"name": "Lentils", "quantity": "250g", "expiry": "2026-09-18"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-09-19"},
            {"name": "Potatoes", "quantity": "4", "expiry": "2026-09-21"},
            {"name": "Spinach", "quantity": "1 bunch", "expiry": "2026-09-22"},
            {"name": "Coconut milk", "quantity": "400ml", "expiry": "2026-09-25"}
        ]
    },

    {
        "serving_size": 3,
        "dietary_restrictions": ["Vegetarian"],
        "allergens": ["None"],
        "ingredients": [
            {"name": "Paneer", "quantity": "400g", "expiry": "2026-09-18"},
            {"name": "Tomatoes", "quantity": "6", "expiry": "2026-09-19"},
            {"name": "Spinach", "quantity": "1 bunch", "expiry": "2026-09-21"},
            {"name": "Potatoes", "quantity": "5", "expiry": "2026-09-23"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-10-01"}
        ]
    },

    {
        "serving_size": 2,
        "dietary_restrictions": ["Vegan"],
        "allergens": ["Nuts"],
        "ingredients": [
            {"name": "Chickpeas", "quantity": "300g", "expiry": "2026-09-17"},
            {"name": "Tomatoes", "quantity": "5", "expiry": "2026-09-18"},
            {"name": "Cabbage", "quantity": "1", "expiry": "2026-09-20"},
            {"name": "Potatoes", "quantity": "4", "expiry": "2026-09-22"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-09-30"}
        ]
    },

    {
        "serving_size": 4,
        "dietary_restrictions": ["Vegetarian"],
        "allergens": ["Nuts"],
        "ingredients": [
            {"name": "Paneer", "quantity": "500g", "expiry": "2026-09-18"},
            {"name": "Yogurt", "quantity": "400g", "expiry": "2026-09-19"},
            {"name": "Tomatoes", "quantity": "8", "expiry": "2026-09-20"},
            {"name": "Onions", "quantity": "6", "expiry": "2026-09-22"},
            {"name": "Rice", "quantity": "1kg", "expiry": "2026-10-01"}
        ]
    },

    {
        "serving_size": 2,
        "dietary_restrictions": ["Vegan"],
        "allergens": ["Soy"],
        "ingredients": [
            {"name": "Lentils", "quantity": "300g", "expiry": "2026-09-18"},
            {"name": "Potatoes", "quantity": "4", "expiry": "2026-09-19"},
            {"name": "Cabbage", "quantity": "1", "expiry": "2026-09-21"},
            {"name": "Tomatoes", "quantity": "5", "expiry": "2026-09-23"},
            {"name": "Rice", "quantity": "500g", "expiry": "2026-09-30"}
        ]
    }
]

print(f"Evaluation test cases: {len(test_cases)}")


# ------------------------------------------------------------
# 2. Instruction used during evaluation
# ------------------------------------------------------------

evaluation_instruction = (
    "Suggest a weekly meal plan for the specified number of people, "
    "prioritizing ingredients that will expire soon. The meal plan must "
    "strictly adhere to the specified dietary restrictions and be free "
    "of all listed allergens. The plan should include three meals per day, "
    "with varied dishes, minimal extra shopping, and leftovers should be "
    "used where appropriate to reduce cooking effort and food waste."
)


# ------------------------------------------------------------
# 3. Generate model response
# ------------------------------------------------------------

def generate_meal_plan(model, tokenizer, test_input):

    messages = [
        {
            "role": "user",
            "content": (
                f"### Instruction: {evaluation_instruction}\n"
                f"### Input: {json.dumps(test_input, ensure_ascii=False)}\n"
                f"### Output:"
            )
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        return_attention_mask=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            use_cache=True,
            do_sample=False
        )

    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


# ------------------------------------------------------------
# 3b. Isolate the model's real answer from an echoed input
# ------------------------------------------------------------
# The model tends to echo "### Input: {...}" / "### Output:" headers
# before its real answer, e.g.:
#   ### Input: {"serving_size": 2, ...}
#   ### Output: {"meals": {...}}
# Every check below must run on the text AFTER the last "### Output:"
# marker, not the raw decoded response - otherwise checks end up
# reading the echoed input JSON (which trivially "contains" every
# pantry ingredient and none of the actual meal plan) instead of what
# the model actually generated.

def get_real_output_text(raw_response):
    marker = raw_response.rfind("### Output:")
    text = raw_response[marker + len("### Output:"):] if marker != -1 else raw_response
    # The model sometimes keeps generating after its real answer ends,
    # producing a truncated repeat of the prompt. skip_special_tokens
    # doesn't always strip this literal marker for this tokenizer, so
    # cut it off manually - anything after it is stale repetition, not
    # part of the real generated meal plan.
    end_marker = text.find("<|endoftext|>")
    if end_marker != -1:
        text = text[:end_marker]
    return text


# ------------------------------------------------------------
# 4. Extract JSON from generated response
# ------------------------------------------------------------

def extract_json(text):

    # First try the entire response
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try to find a JSON object
    start = text.find("{")

    if start == -1:
        return None

    # Find matching closing brace
    brace_count = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):

        char = text[i]

        if escape:
            escape = False
            continue

        if char == "\\":
            escape = True
            continue

        if char == '"':
            in_string = not in_string

        if not in_string:

            if char == "{":
                brace_count += 1

            elif char == "}":
                brace_count -= 1

                if brace_count == 0:

                    candidate = text[start:i+1]

                    try:
                        return json.loads(candidate)
                    except Exception:
                        return None

    return None


# ------------------------------------------------------------
# 5. Ingredient knowledge for objective checking
# ------------------------------------------------------------

ingredient_properties = {

    # Vegan / vegetarian
    "chicken": {"vegan": False, "vegetarian": False, "soy": False,
                "dairy": False, "gluten": False, "nuts": False},

    "mutton": {"vegan": False, "vegetarian": False, "soy": False,
               "dairy": False, "gluten": False, "nuts": False},

    "fish": {"vegan": False, "vegetarian": False, "soy": False,
             "dairy": False, "gluten": False, "nuts": False},

    "egg": {"vegan": False, "vegetarian": False, "soy": False,
            "dairy": False, "gluten": False, "nuts": False},

    "eggs": {"vegan": False, "vegetarian": False, "soy": False,
             "dairy": False, "gluten": False, "nuts": False},

    "paneer": {"vegan": False, "vegetarian": True, "soy": False,
               "dairy": True, "gluten": False, "nuts": False},

    "cheese": {"vegan": False, "vegetarian": True, "soy": False,
               "dairy": True, "gluten": False, "nuts": False},

    "milk": {"vegan": False, "vegetarian": True, "soy": False,
             "dairy": True, "gluten": False, "nuts": False},

    "yogurt": {"vegan": False, "vegetarian": True, "soy": False,
               "dairy": True, "gluten": False, "nuts": False},

    # Common vegan foods
    "tofu": {"vegan": True, "vegetarian": True, "soy": True,
             "dairy": False, "gluten": False, "nuts": False},

    "lentils": {"vegan": True, "vegetarian": True, "soy": False,
                "dairy": False, "gluten": False, "nuts": False},

    "chickpeas": {"vegan": True, "vegetarian": True, "soy": False,
                  "dairy": False, "gluten": False, "nuts": False},

    "rice": {"vegan": True, "vegetarian": True, "soy": False,
             "dairy": False, "gluten": False, "nuts": False},

    "rice flour": {"vegan": True, "vegetarian": True, "soy": False,
                   "dairy": False, "gluten": False, "nuts": False},

    "potatoes": {"vegan": True, "vegetarian": True, "soy": False,
                 "dairy": False, "gluten": False, "nuts": False},

    "spinach": {"vegan": True, "vegetarian": True, "soy": False,
                "dairy": False, "gluten": False, "nuts": False},

    "cabbage": {"vegan": True, "vegetarian": True, "soy": False,
                "dairy": False, "gluten": False, "nuts": False},

    "tomatoes": {"vegan": True, "vegetarian": True, "soy": False,
                 "dairy": False, "gluten": False, "nuts": False},

    "onions": {"vegan": True, "vegetarian": True, "soy": False,
               "dairy": False, "gluten": False, "nuts": False},

    "bell peppers": {"vegan": True, "vegetarian": True, "soy": False,
                     "dairy": False, "gluten": False, "nuts": False},

    "coconut milk": {"vegan": True, "vegetarian": True, "soy": False,
                     "dairy": False, "gluten": False, "nuts": False},

    "coconut": {"vegan": True, "vegetarian": True, "soy": False,
                "dairy": False, "gluten": False, "nuts": False},

    "bread": {"vegan": True, "vegetarian": True, "soy": False,
              "dairy": False, "gluten": True, "nuts": False},

    "roti": {"vegan": True, "vegetarian": True, "soy": False,
             "dairy": False, "gluten": True, "nuts": False},

    "wheat": {"vegan": True, "vegetarian": True, "soy": False,
              "dairy": False, "gluten": True, "nuts": False},

    "flour": {"vegan": True, "vegetarian": True, "soy": False,
              "dairy": False, "gluten": True, "nuts": False},

    "nuts": {"vegan": True, "vegetarian": True, "soy": False,
             "dairy": False, "gluten": False, "nuts": True},

    "peanuts": {"vegan": True, "vegetarian": True, "soy": False,
                "dairy": False, "gluten": False, "nuts": True}
}


# ------------------------------------------------------------
# 5b. Strip dietary/allergen compliance disclaimers
# ------------------------------------------------------------
# Training data descriptions end with a disclaimer like
# "...This is a Vegan and Nuts free meal." Since "nuts" is also a real
# ingredient key above, naive matching flags every compliant meal as
# containing nuts just because it CLAIMS to be nut-free. Strip these
# disclaimer phrases before scanning for real ingredient mentions.

_COMPLIANCE_PATTERN = re.compile(
    r"\b(vegan|vegetarian|gluten[- ]?free|dairy[- ]?free|nuts?[- ]?free|soy[- ]?free)\b",
    re.IGNORECASE,
)

# A modifier like "gluten-free" or "non-dairy" appearing directly before
# an ingredient mention overrides that ONE property for that mention -
# e.g. "gluten-free bread" is not actually gluten-containing, even
# though "bread" normally is. Checked per-mention, not stripped globally,
# because a mention without the modifier should still be flagged.
_NEGATION_MODIFIERS = {
    "gluten": re.compile(r"(gluten[- ]?free)\s*$", re.IGNORECASE),
    "dairy": re.compile(r"(dairy[- ]?free|non[- ]?dairy)\s*$", re.IGNORECASE),
    "soy": re.compile(r"(soy[- ]?free)\s*$", re.IGNORECASE),
    "nuts": re.compile(r"(nuts?[- ]?free)\s*$", re.IGNORECASE),
}


def _is_negated_at(text, match_start, property_name):
    pattern = _NEGATION_MODIFIERS.get(property_name)
    if not pattern:
        return False
    window = text[max(0, match_start - 20):match_start]
    return bool(pattern.search(window))


def _find_ingredient_mentions(text):
    """Yield (ingredient, match_start) for every mention, longest names
    first so compounds like 'coconut milk' are consumed before 'milk'
    can match inside them. Matched spans are blanked (not deleted) so
    positions stay valid for the negation-window check."""
    remaining = text
    mentions = []
    for ingredient in sorted(ingredient_properties, key=len, reverse=True):
        pattern = r'\b' + re.escape(ingredient) + r'\b'
        for m in re.finditer(pattern, remaining):
            mentions.append((ingredient, m.start()))
        remaining = re.sub(pattern, lambda mm: ' ' * len(mm.group()), remaining)
    return mentions


# ------------------------------------------------------------
# 6. Find ingredients mentioned in a meal description
# ------------------------------------------------------------

def find_mentioned_ingredients(text):

    text = text.lower()
    text = _COMPLIANCE_PATTERN.sub("", text)

    found = []

    # Check longer/compound ingredient names first (e.g. "coconut milk")
    # and remove matched text as we go, so a shorter name like "milk"
    # doesn't falsely match inside a compound ingredient it's part of.
    for ingredient in sorted(ingredient_properties, key=len, reverse=True):

        pattern = r'\b' + re.escape(ingredient) + r'\b'

        if re.search(pattern, text):
            found.append(ingredient)
            text = re.sub(pattern, ' ', text)

    return found


# ------------------------------------------------------------
# 7. Dietary compliance
# ------------------------------------------------------------

def check_dietary_compliance(output_text, restrictions):

    text = output_text.lower()
    text = _COMPLIANCE_PATTERN.sub("", text)

    mentions = _find_ingredient_mentions(text)

    violations = []

    for ingredient, start in mentions:

        props = ingredient_properties[ingredient]

        for restriction in restrictions:

            restriction_lower = restriction.lower()

            if restriction_lower not in ("vegan", "vegetarian"):
                continue

            if props[restriction_lower]:
                continue

            # e.g. "milk" fails vegan only because of dairy content -
            # if this specific mention is explicitly "non-dairy", it's
            # not actually the dairy version.
            if props.get("dairy") and _is_negated_at(text, start, "dairy"):
                continue

            violations.append(ingredient)

    return len(set(violations)) == 0, list(set(violations))


# ------------------------------------------------------------
# 8. Allergen compliance
# ------------------------------------------------------------

def check_allergen_compliance(output_text, allergens):

    if not allergens or allergens == ["None"]:
        return True, []

    text = output_text.lower()
    text = _COMPLIANCE_PATTERN.sub("", text)

    mentions = _find_ingredient_mentions(text)

    violations = []

    for ingredient, start in mentions:

        props = ingredient_properties[ingredient]

        for allergen in allergens:

            allergen_lower = allergen.lower()

            if allergen_lower not in props or not props[allergen_lower]:
                continue

            if _is_negated_at(text, start, allergen_lower):
                continue

            violations.append(ingredient)

    return len(set(violations)) == 0, list(set(violations))


# ------------------------------------------------------------
# 9. Meal-plan completeness
# ------------------------------------------------------------

days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

meal_types = [
    "breakfast",
    "lunch",
    "dinner"
]


def check_completeness(parsed_output):

    if not isinstance(parsed_output, dict):
        return 0, 21

    meals = parsed_output.get("meals", {})

    if not isinstance(meals, dict):
        return 0, 21

    completed = 0

    for day in days:

        day_data = meals.get(day, {})

        if not isinstance(day_data, dict):
            continue

        for meal in meal_types:

            meal_data = day_data.get(meal)

            if isinstance(meal_data, dict):

                description = meal_data.get("display_text", "")

                if description and str(description).strip():
                    completed += 1

    return completed, 21


# ------------------------------------------------------------
# 10. Pantry utilization
# ------------------------------------------------------------

def calculate_pantry_utilization(output_text, pantry):

    text = output_text.lower()

    used = []

    for item in pantry:

        name = item["name"].lower()

        # Handle common naming variations
        variants = [name]

        if name == "lentils":
            variants += ["toor dal", "dal"]

        elif name == "rice":
            variants += ["rice flour"]

        for variant in variants:

            if variant in text:
                used.append(name)
                break

    used = list(set(used))

    total = len(pantry)

    if total == 0:
        return 0, 0, 0

    percentage = (len(used) / total) * 100

    return percentage, len(used), total


# ------------------------------------------------------------
# 11. Near-expiry utilization
# ------------------------------------------------------------

def calculate_expiry_utilization(parsed_output, pantry, n=3, days_window=2):
    """Checks whether the n soonest-expiring pantry items appear within
    the first `days_window` days of the plan - NOT anywhere in the whole
    week. Using an ingredient on day 7 is not prioritizing it; scanning
    the whole week (as the old version did) made this metric almost
    identical to plain pantry utilization, which is why it was always
    ~100% regardless of whether the model actually prioritized anything."""

    if not isinstance(parsed_output, dict):
        return 0, 0, 0

    meals = parsed_output.get("meals", {})

    if not isinstance(meals, dict):
        return 0, 0, 0

    sorted_pantry = sorted(
        pantry,
        key=lambda x: datetime.strptime(
            x["expiry"], "%Y-%m-%d"
        )
    )

    priority_items = sorted_pantry[:min(n, len(sorted_pantry))]

    early_days = days[:days_window]
    early_text_parts = []

    for day in early_days:

        day_data = meals.get(day, {})

        if not isinstance(day_data, dict):
            continue

        for meal in meal_types:

            meal_data = day_data.get(meal)

            if isinstance(meal_data, dict):
                early_text_parts.append(str(meal_data.get("description_for_model", "")))
                early_text_parts.append(str(meal_data.get("display_text", "")))

    early_text = " ".join(early_text_parts).lower()

    used = []

    for item in priority_items:

        name = item["name"].lower()

        variants = [name]

        if name == "lentils":
            variants += ["toor dal", "dal"]

        for variant in variants:

            if variant in early_text:
                used.append(name)
                break

    total = len(priority_items)

    if total == 0:
        return 0, 0, 0

    percentage = (len(set(used)) / total) * 100

    return percentage, len(set(used)), total


# ------------------------------------------------------------
# 12. Evaluate model
# ------------------------------------------------------------

results = []

print("\nStarting evaluation...\n")

for i, test_input in enumerate(test_cases, start=1):

    print(f"Evaluating test case {i}/{len(test_cases)}...")

    response = generate_meal_plan(
        model,
        tokenizer,
        test_input
    )

    # Isolate the real generated answer (strips any echoed input)
    real_output_text = get_real_output_text(response)

    parsed = extract_json(real_output_text)

    # -----------------------------
    # Output validity
    # -----------------------------

    valid_json = parsed is not None

    # -----------------------------
    # Completeness
    # -----------------------------

    completed_slots, total_slots = check_completeness(parsed)

    completeness = (
        completed_slots / total_slots * 100
        if total_slots > 0 else 0
    )

    # -----------------------------
    # Dietary compliance
    # -----------------------------

    dietary_ok, dietary_violations = check_dietary_compliance(
        real_output_text,
        test_input["dietary_restrictions"]
    )

    # -----------------------------
    # Allergen compliance
    # -----------------------------

    allergen_ok, allergen_violations = check_allergen_compliance(
        real_output_text,
        test_input["allergens"]
    )

    # -----------------------------
    # Pantry utilization
    # -----------------------------

    pantry_utilization, pantry_used, pantry_total = \
        calculate_pantry_utilization(
            real_output_text,
            test_input["ingredients"]
        )

    # -----------------------------
    # Expiry utilization
    # -----------------------------

    expiry_utilization, expiry_used, expiry_total = \
        calculate_expiry_utilization(
            parsed,
            test_input["ingredients"],
            n=3
        )

    # -----------------------------
    # Overall constraint satisfaction
    # -----------------------------

    all_constraints_satisfied = (
        valid_json
        and completeness == 100
        and dietary_ok
        and allergen_ok
    )

    results.append({
        "Test Case": i,
        "Valid Output": valid_json,
        "Completed Slots": completed_slots,
        "Total Slots": total_slots,
        "Completeness (%)": round(completeness, 2),
        "Dietary Compliance": dietary_ok,
        "Dietary Violations": ", ".join(dietary_violations),
        "Allergen Compliance": allergen_ok,
        "Allergen Violations": ", ".join(allergen_violations),
        "Pantry Utilization (%)": round(pantry_utilization, 2),
        "Pantry Ingredients Used": pantry_used,
        "Pantry Ingredients Total": pantry_total,
        "Near-Expiry Utilization (%)": round(expiry_utilization, 2),
        "Near-Expiry Used": expiry_used,
        "Near-Expiry Total": expiry_total,
        "Overall Constraint Satisfaction": all_constraints_satisfied,
        "Raw Output": response,               # full raw text, kept for debugging
        "Real Output (used for checks)": real_output_text,
    })


# ------------------------------------------------------------
# 13. Create evaluation dataframe
# ------------------------------------------------------------

results_df = pd.DataFrame(results)


# ------------------------------------------------------------
# 14. Summary metrics
# ------------------------------------------------------------

num_cases = len(results_df)

valid_output_rate = (
    results_df["Valid Output"].mean() * 100
)

dietary_compliance_rate = (
    results_df["Dietary Compliance"].mean() * 100
)

allergen_compliance_rate = (
    results_df["Allergen Compliance"].mean() * 100
)

overall_constraint_rate = (
    results_df["Overall Constraint Satisfaction"].mean() * 100
)

average_completeness = (
    results_df["Completeness (%)"].mean()
)

average_pantry_utilization = (
    results_df["Pantry Utilization (%)"].mean()
)

average_expiry_utilization = (
    results_df["Near-Expiry Utilization (%)"].mean()
)


# ------------------------------------------------------------
# 15. Evaluation matrix
# ------------------------------------------------------------

evaluation_matrix = pd.DataFrame({

    "Metric": [
        "Valid Output",
        "Meal-Plan Completeness",
        "Dietary Compliance",
        "Allergen Compliance",
        "Pantry Utilization",
        "Near-Expiry Utilization",
        "Overall Constraint Satisfaction"
    ],

    "Score (%)": [
        round(valid_output_rate, 2),
        round(average_completeness, 2),
        round(dietary_compliance_rate, 2),
        round(allergen_compliance_rate, 2),
        round(average_pantry_utilization, 2),
        round(average_expiry_utilization, 2),
        round(overall_constraint_rate, 2)
    ]
})


# ------------------------------------------------------------
# 16. Print final results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SMARTPANTRY MODEL EVALUATION")
print("=" * 60)

print(f"\nNumber of test cases: {num_cases}")

print("\nEvaluation Matrix:")
display(evaluation_matrix)

print("\nDetailed Results:")
display(
    results_df[
        [
            "Test Case",
            "Valid Output",
            "Completeness (%)",
            "Dietary Compliance",
            "Allergen Compliance",
            "Pantry Utilization (%)",
            "Near-Expiry Utilization (%)",
            "Overall Constraint Satisfaction"
        ]
    ]
)


# ------------------------------------------------------------
# 17. Save results
# ------------------------------------------------------------

evaluation_matrix.to_csv(
    "smartpantry_evaluation_matrix.csv",
    index=False
)

results_df.to_csv(
    "smartpantry_detailed_evaluation.csv",
    index=False
)

print("\nResults saved:")
print("  - smartpantry_evaluation_matrix.csv")
print("  - smartpantry_detailed_evaluation.csv")

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluation test cases: 10

Starting evaluation...

Evaluating test case 1/10...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating test case 2/10...


Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Evaluating test case 10/10...

SMARTPANTRY MODEL EVALUATION

Number of test cases: 10

Evaluation Matrix:


,Metric,Score (%)
0,Valid Output,100.00
1,Meal-Plan Completeness,100.00
2,Dietary Compliance,100.00
3,Allergen Compliance,90.00
4,Pantry Utilization,96.67
5,Near-Expiry Utilization,93.33
6,Overall Constraint Satisfaction,90.00



Detailed Results:


,Test Case,Valid Output,Completeness (%),Dietary Compliance,Allergen Compliance,Pantry Utilization (%),Near-Expiry Utilization (%),Overall Constraint Satisfaction
0,1,True,100.0,True,True,100.00,100.00,True
1,2,True,100.0,True,True,66.67,66.67,True
2,3,True,100.0,True,True,100.00,100.00,True
3,4,True,100.0,True,True,100.00,100.00,True
4,5,True,100.0,True,True,100.00,100.00,True
5,6,True,100.0,True,False,100.00,100.00,False
6,7,True,100.0,True,True,100.00,100.00,True
7,8,True,100.0,True,True,100.00,66.67,True
8,9,True,100.0,True,True,100.00,100.00,True
9,10,True,100.0,True,True,100.00,100.00,True



Results saved:
  - smartpantry_evaluation_matrix.csv
  - smartpantry_detailed_evaluation.csv


In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt (dataset format)
test_instruction = "Suggest a weekly meal plan for 2 people, prioritizing ingredients that will expire soon. The meal plan must strictly adhere to the specified dietary restrictions for example vegans cannot have any meant products in their diet, and be free of all listed allergens. The plan should include three meals per day, with varied dishes, and minimal extra shopping. Leftovers should be used to reduce cooking effort and food waste. Also include popular Western dishes available in India and Kerala dishes."

test_input = {
    "serving_size": 2,
    "dietary_restrictions": [
        "Vegan"
    ],
    "allergens": [
        "Nuts"
    ],
    "ingredients": [
        {
            "name": "Lentils (Toor Dal)",
            "quantity": "250g",
            "expiry": "2025-09-02"
        },
        {
            "name": "Tofu",
            "quantity": "200g",
            "expiry": "2025-09-03"
        },
        {
            "name": "Spinach",
            "quantity": "1 bunch",
            "expiry": "2025-09-04"
        },
        {
            "name": "Cabbage",
            "quantity": "1 medium head",
            "expiry": "2025-09-07"
        },
        {
            "name": "Potatoes",
            "quantity": "4 medium",
            "expiry": "2025-09-10"
        },
        {
            "name": "Rice Flour",
            "quantity": "300g",
            "expiry": "2025-10-01"
        }
    ]
}

# The messages list must have a "content" key
# We will use the format you fine-tuned the model on.
messages = [
    {"role": "user", "content": f"### Instruction: {test_instruction}\n### Input: {json.dumps(test_input, ensure_ascii=False)}\n### Output:"}
]

# Step 1: Get chat-formatted text
chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,              # <-- don't tokenize yet
    add_generation_prompt=True
)

# Step 2: Tokenize with attention_mask
inputs = tokenizer(
    chat_text,
    return_tensors="pt",
    return_attention_mask=True, # <-- ensures attention_mask is included
).to("cuda")

# Step 3: Generate response
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
response = response.split("<|endoftext|>")[0]
print(response)

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 16 Sep 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

### Instruction: Suggest a weekly meal plan for 2 people, prioritizing ingredients that will expire soon. The meal plan must strictly adhere to the specified dietary restrictions for example vegans cannot have any meant products in their diet, and be free of all listed allergens. The plan should include three meals per day, with varied dishes, and minimal extra shopping. Leftovers should be used to reduce cooking effort and food waste. Also include popular Western dishes available in India and Kerala dishes.
### Input: {"serving_size": 2, "dietary_restrictions": ["Vegan"], "allergens": ["Nuts"], "ingredients": [{"name": "Lentils (Toor Dal)", "quantity": "250g", "expiry": "2025-09-02"}, {"name": "Tofu", "quantity": "200g", "expiry": "2025-09-03"}, {"name": "Spinach", "quantity": "1 bunch", "expiry

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

base_model_id = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
adapter_path = "outputs/checkpoint-168"   # replace with your latest checkpoint

# Save tokenizer (it’s from the base model)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained("meal-gen-adapter")

# Just copy adapter folder (LoRA) for upload
!cp -r {adapter_path} meal-gen-adapter/


cp: cannot stat 'outputs/checkpoint-168': No such file or directory


In [ ]:
!ls meal-gen-adapter


chat_template.jinja  tokenizer_config.json  tokenizer.json


In [ ]:
from transformers import AutoTokenizer

base_model_id = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained("meal-gen-adapter")


('meal-gen-adapter/tokenizer_config.json',
 'meal-gen-adapter/chat_template.jinja',
 'meal-gen-adapter/tokenizer.json')